In [ ]:
# hide
import numpy as np
import pyquist as pq
from pyquist.helper import amplitude_to_db, db_to_amplitude

In [ ]:
# collapse
def rms_envelope(x: np.ndarray, frame_length: int, hop: int) -> np.ndarray:
    """The RMS amplitude of each frame of x, centered on samples 0, hop, 2 * hop, ..."""
    padded = np.pad(x, frame_length // 2)
    frames = np.lib.stride_tricks.sliding_window_view(padded, frame_length)[::hop]
    return np.sqrt(np.mean(frames ** 2, axis=1))

In [ ]:
def compress(x: np.ndarray, threshold_db: float, ratio: float,
             key: np.ndarray | None = None, f_s: int = 44100) -> np.ndarray:
    """Turn x down wherever the level of `key` (x itself, by default) exceeds
    the threshold, so that the level above the threshold shrinks by `ratio`."""
    key = x if key is None else key
    hop = f_s // 100                                     # 100 frames per second
    level_db = amplitude_to_db(np.maximum(rms_envelope(key, 4 * hop, hop), 1e-6))
    target_db = np.where(level_db > threshold_db,
                         threshold_db + (level_db - threshold_db) / ratio, level_db)
    gain = db_to_amplitude(target_db - level_db)         # one gain per frame...
    return x * np.interp(np.arange(len(x)), hop * np.arange(len(gain)), gain)  # ...per sample


f_s = 44100
beethoven = pq.Audio.from_file("../assets/beethoven.wav").samples[:, 0]
compressed = compress(beethoven, threshold_db=-30.0, ratio=4.0)
pq.play(pq.Audio(beethoven, f_s), normalize=True)
pq.play(pq.Audio(compressed, f_s), normalize=True)   # normalizing restores the peak level